# 高斯混合模型（GMM）算法推导（完整版）
---
---



## 1. 模型定义（区分单个样本和全体样本）

设数据集包含 $N$ 个样本 $\mathbf{X} = \{x_1, x_2, \dots, x_N\}$，每个样本 $x_i \in \mathbb{R}^d$。GMM 假设数据由 $K$ 个高斯分布混合生成：

### 单个样本的概率密度
$$
p(x_i \mid \theta) = \sum_{k=1}^K \pi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)
$$
其中：
- $\pi_k$：第 $k$ 个高斯分布的**混合系数**（$\sum_{k=1}^K \pi_k = 1$）
- $\mathcal{N}(x_i \mid \mu_k, \Sigma_k)$：高斯分布的概率密度函数：
  $$
  \mathcal{N}(x_i \mid \mu_k, \Sigma_k) = \frac{1}{(2\pi)^{d/2} |\Sigma_k|^{1/2}} \exp\left(-\frac{1}{2}(x_i - \mu_k)^T \Sigma_k^{-1} (x_i - \mu_k)\right)
  $$

### 全体样本的联合概率（假设独立同分布）
$$
p(\mathbf{X} \mid \theta) = \prod_{i=1}^N p(x_i \mid \theta) = \prod_{i=1}^N \left( \sum_{k=1}^K \pi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right)
$$

---





## 2. 引入隐变量（明确样本归属）

定义隐变量 $z_{ik} \in \{0,1\}$，表示样本 $x_i$ 是否属于第 $k$ 个高斯分布（1-hot 编码）：
- **隐变量先验**：$p(z_{ik} = 1) = \pi_k$
- **条件概率**：$p(x_i \mid z_{ik} = 1) = \mathcal{N}(x_i \mid \mu_k, \Sigma_k)$

### 完全数据的联合概率（显式写出样本下标）
$$
p(\mathbf{X}, \mathbf{Z} \mid \theta) = \prod_{i=1}^N \prod_{k=1}^K \left[ \pi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}}
$$

### 上面这个联合概率公式的推导过程：

### 2.1 模型设定与符号说明

#### 观测数据
- $\mathbf{X} = \{x_1, x_2, \dots, x_N\}$
- 每个样本 $x_i \in \mathbb{R}^d$ 是第 $i$ 个观测数据

#### 隐变量
- $\mathbf{Z} = \{z_{ik}\}$
- $z_{ik} \in \{0,1\}$ （1-hot编码）
  - 表示样本 $x_i$ 是否属于第 $k$ 个高斯成分

#### 模型参数
- $\theta = \{\pi_k, \mu_k, \Sigma_k\}_{k=1}^K$
  - $\pi_k$: 第 $k$ 个高斯成分的混合系数 ($\sum_{k=1}^K \pi_k = 1$)
  - $\mu_k, \Sigma_k$: 第 $k$ 个高斯分布的均值和协方差矩阵

### 2.2 联合概率分解

联合概率可以分解为：
$$
p(\mathbf{X},\mathbf{Z}|\theta) = p(\mathbf{Z}|\theta) \cdot p(\mathbf{X}|\mathbf{Z},\theta)
$$

#### (1) 隐变量的先验分布
$$
p(\mathbf{Z}|\theta) = \prod_{i=1}^N \prod_{k=1}^K \pi_k^{z_{ik}}
$$

#### (2) 观测数据的条件分布
$$
p(\mathbf{X}|\mathbf{Z},\theta) = \prod_{i=1}^N \prod_{k=1}^K \mathcal{N}(x_i|\mu_k,\Sigma_k)^{z_{ik}}
$$

### 2.3 联合概率的最终形式

将两部分相乘，并利用 $z_{ik}$ 的指示性质：
$$
p(\mathbf{X},\mathbf{Z}|\theta) = \prod_{i=1}^N \prod_{k=1}^K [\pi_k \mathcal{N}(x_i|\mu_k,\Sigma_k)]^{z_{ik}}
$$

#### 关键点说明
1. **乘积符号**：
   - 外层 $\prod_{i=1}^N$：所有样本独立同分布（i.i.d.）
   - 内层 $\prod_{k=1}^K$：每个样本只能属于一个高斯成分（由 $z_{ik}$ 控制）

2. **指示变量 $z_{ik}$ 的作用**：
   - 当 $z_{ik}=1$ 时：激活对应的高斯成分
   - 当 $z_{ik}=0$ 时：该项退化为1（$a^0=1$），不影响乘积结果

### 2.4 直观解释

#### 单样本视角
对于单个样本 $x_i$，假设 $z_{ic}=1$（属于第 $c$ 个成分），其他 $z_{ik}=0$：
$$
p(x_i,z_{i1}=0,\dots,z_{ic}=1,\dots,z_{iK}=0|\theta) = \pi_c \mathcal{N}(x_i|\mu_c,\Sigma_c)
$$

#### 全局视角
所有样本的联合概率是各样本对应成分概率的乘积，体现了独立同分布假设。

### 2.5 与EM算法的关联

该联合概率公式是EM算法的基础：

1. **E步**：
   - 计算隐变量的后验期望 $\gamma_{ik} = \mathbb{E}[z_{ik}|x_i,\theta]$

2. **M步**：
   - 通过最大化联合概率的期望更新参数 $\theta$

### 2.6 总结

推导逻辑：
1. 用 $z_{ik}$ 表示样本归属（多项式分布）
2. 给定隐变量，观测数据服从高斯分布
3. 样本之间、隐变量之间相互独立
4. 联合概率 = 先验 × 似然，用 $z_{ik}$ 控制激活项

该公式是GMM的核心，为后续EM算法参数估计提供了数学基础。

---

## 3. EM算法推导（分步说明）

### (1) E步：计算后验概率 $\gamma_{ik}$
对每个样本 $x_i$，计算其属于第 $k$ 个高斯分布的后验概率（责任值）：
$$
\gamma_{ik} = p(z_{ik} = 1 \mid x_i, \theta^{(t)}) = \frac{\pi_k^{(t)} \mathcal{N}(x_i \mid \mu_k^{(t)}, \Sigma_k^{(t)})}{\sum_{j=1}^K \pi_j^{(t)} \mathcal{N}(x_i \mid \mu_j^{(t)}, \Sigma_j^{(t)})}
$$
**物理意义**：样本 $x_i$ 对第 $k$ 个高斯成分的"贡献度"。

**上标$t$ 的含义**：  
  $t$ 是 **EM算法（期望最大化算法）的当前迭代步数**。具体说明：
  
  - **初始化**（$t=0$）:  
    $\theta^{(0)} = \{\pi_k^{(0)}, \mu_k^{(0)}, \Sigma_k^{(0)}\}$  
  - **迭代过程**（$t=1,2,\dots$）:  
    每次迭代更新参数，记为 $\theta^{(t)}$。

**动态更新规则**：  
  在每次迭代中，参数会重新计算并更新为：$\theta^{(t+1)}$, 直到收敛（如似然函数不再显著变化）。

**参数依赖与迭代说明**：
- 计算责任值 $\gamma_{ik}$ 时，使用的是**当前迭代步 $t$ 的参数**：
  $$
  \gamma_{ik} = \frac{\pi_k^{(t)} \mathcal{N}(x_i \mid \mu_k^{(t)}, \Sigma_k^{(t)})}{\sum_{j=1}^K \pi_j^{(t)} \mathcal{N}(x_i \mid \mu_j^{(t)}, \Sigma_j^{(t)})}
  $$
  - **注意**：依赖的是临时参数 $\theta^{(t)}$，而非最终参数。

**迭代优化需求**
- EM算法通过交替执行以下步骤逐步优化：
1. **E步**（Expectation）：
   - 使用当前参数 $\theta^{(t)}$ 计算隐变量期望 $\gamma_{ik}$。
2. **M步**（Maximization）：
   - 用 $\gamma_{ik}$ 更新参数，得到 $\theta^{(t+1)}$。

**避免混淆**
- 上标 $(t)$ 明确标识**迭代阶段**，确保：
  - E步计算时使用当前参数 $\theta^{(t)}$。
  - M步生成的是下一轮参数 $\theta^{(t+1)}$。



### (2) M步：更新参数（显式分离样本和参数）
#### (2.1) 最大化完全数据的对数似然期望 $Q(\theta, \theta^{(t)})$：
$$
Q(\theta, \theta^{(t)}) = \sum_{i=1}^N \sum_{k=1}^K \gamma_{ik} \left[ \log \pi_k + \log \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]
$$

##### 这个公式是怎么推导出来的: EM算法原理与GMM推导

##### (2.1.1) EM算法的基本思想
EM算法用于在含有隐变量（如GMM中的分量标记$z_i$）的情况下进行最大似然估计（MLE）。我们不能直接最大化观测数据的对数似然：

$$
\log p(X|\theta) = \sum_{i=1}^N \log \sum_{z_i} p(x_i,z_i|\theta)
$$

因为包含了对隐变量的求和，对数和在一起很难直接最大化。

##### (2.1.2) 引入下界：使用Jensen不等式
对于每个样本$x_i$，我们有：

$$
\log p(x_i|\theta) = \log \sum_{z_i} p(x_i,z_i|\theta)
$$

引入一个任意分布$q(z_i)$，利用Jensen不等式（凸函数在期望外大于期望值）：

$$
\log \sum_{z_i} p(x_i,z_i|\theta) = \log \sum_{z_i} q(z_i) \cdot \frac{p(x_i,z_i|\theta)}{q(z_i)} \geq \sum_{z_i} q(z_i) \log \frac{p(x_i,z_i|\theta)}{q(z_i)}
$$

对全部样本求和：

$$
\log p(X|\theta) \geq \sum_{i=1}^N \sum_{z_i} q(z_i) \log \frac{p(x_i,z_i|\theta)}{q(z_i)} =: \mathcal{L}(q,\theta)
$$

这个右侧就是**变分下界**。EM的策略就是：
1. **E步**：用旧参数$\theta^{old}$固定$q(z_i)=p(z_i|x_i,\theta^{old})$
2. **M步**：最大化该下界$\mathcal{L}(q,\theta)$关于$\theta$

##### (2.1.3) 推导M步目标函数$Q(\theta,\theta^{old})$
由于我们选择：

$$
q(z_i) = p(z_i|x_i,\theta^{old}) = \gamma_{ik}
$$

将下界写成两部分：

$$
\mathcal{L}(q,\theta) = \sum_{i=1}^N \sum_{z_i} q(z_i) \log p(x_i,z_i|\theta) - \sum_{i=1}^N \sum_{z_i} q(z_i) \log q(z_i)
$$

记第一项为：

$$
Q(\theta,\theta^{old}) := \sum_{i=1}^N \sum_{z_i} p(z_i|x_i,\theta^{old}) \log p(x_i,z_i|\theta)
$$

**这就是我们在M步中要最大化的期望对数似然函数。**

为什么这里我们只用记第一项？

我们看一下EM算法中的变分下界分析

EM算法最大化的是观测数据对数似然的变分下界：

$$
\log p(X|\theta) \geq \mathcal{L}(q,\theta) = \underbrace{\sum_{i=1}^N \sum_{z_i} q(z_i) \log p(x_i,z_i|\theta)}_{\text{期望对数似然 } Q(\theta,\theta^{old})} - \underbrace{\sum_{i=1}^N \sum_{z_i} q(z_i) \log q(z_i)}_{\text{熵项}}
$$

也可以表示为：

$$
\mathcal{L}(q,\theta) = Q(\theta,\theta^{old}) + H(q)
$$

其中：
- $Q(\theta,\theta^{old}) = \mathbb{E}_{q(z)}[\log p(x,z|\theta)]$
- $H(q) = -\mathbb{E}_{q(z)}[\log q(z)]$（熵项，仅与$q(z)$有关）

为什么M步可以忽略熵项？

关键原因是，在M步中，$q(z)$是固定的：
1. **E步**：固定旧参数$\theta^{old}$，设置
   $$
   q(z_i) := p(z_i|x_i,\theta^{old}) = \gamma_{ik}
   $$
2. **M步**：固定$q(z)$，最大化$\mathcal{L}(q,\theta)$关于$\theta$

由于熵项$H(q)$完全由固定的$q(z_i)$决定，与$\theta$无关，因此在M步中它是一个常数，可以忽略。

总结
M步中只需最大化期望对数似然$Q(\theta,\theta^{old})$，因为：
1. 变分下界中的熵项$H(q)$不依赖于$\theta$
2. 在M步中熵项为常数
3. 最大化$\mathcal{L}(q,\theta)$等价于最大化$Q(\theta,\theta^{old})$






##### (2.1.4) 应用到GMM的情形
在GMM中，$z_i \in \{1,\dots,K\}$，是one-hot编码的离散变量，所以：

$$
p(x_i,z_i=k|\theta) = \pi_k \mathcal{N}(x_i|\mu_k,\Sigma_k)
$$

于是：

$$
Q(\theta,\theta^{old}) = \sum_{i=1}^N \sum_{k=1}^K \gamma_{ik} \log [\pi_k \mathcal{N}(x_i|\mu_k,\Sigma_k)]
$$

展开为：

$$
Q(\theta,\theta^{old}) = \sum_{i=1}^N \sum_{k=1}^K \gamma_{ik} [\log \pi_k + \log \mathcal{N}(x_i|\mu_k,\Sigma_k)]
$$

##### (2.1.5) 总结
M步的目标函数：

$$
Q(\theta,\theta^{old}) = \sum_{i=1}^N \sum_{k=1}^K \gamma_{ik} [\log \pi_k + \log \mathcal{N}(x_i|\mu_k,\Sigma_k)]
$$

是通过：
1. 使用Jensen不等式对对数似然建立下界
2. 在E步中设置$q(z_i)=p(z_i|x_i,\theta^{old})$
3. 在M步中最大化该下界中关于$\theta$的部分



#### (2.2)更新混合系数 $\pi_k$
$$
\pi_k^{(t+1)} = \frac{1}{N} \sum_{i=1}^N \gamma_{ik}
$$
**解释**：所有样本对第 $k$ 个成分的责任值均值。

#### (2.3)更新均值 $\mu_k$
$$
\mu_k^{(t+1)} = \frac{\sum_{i=1}^N \gamma_{ik} x_i}{\sum_{i=1}^N \gamma_{ik}}
$$
**解释**：样本 $x_i$ 的加权平均（权重为 $\gamma_{ik}$）。

#### (2.4)更新协方差 $\Sigma_k$
$$
\Sigma_k^{(t+1)} = \frac{\sum_{i=1}^N \gamma_{ik} (x_i - \mu_k^{(t+1)}) (x_i - \mu_k^{(t+1)})^T}{\sum_{i=1}^N \gamma_{ik}}
$$
**解释**：样本 $x_i$ 的加权协方差。

---


## 4. 算法流程总结

1. **初始化**：随机设定 $\theta^{(0)} = \{\pi_k, \mu_k, \Sigma_k\}_{k=1}^K$
2. **迭代直至收敛**：
   - **E步**：对每个样本 $x_i$，计算 $\gamma_{ik}$
   - **M步**：用所有样本的 $\gamma_{ik}$ 更新 $\pi_k, \mu_k, \Sigma_k$
3. **输出**：
   - 参数 $\theta$
   - 样本 $x_i$ 的聚类标签 $\arg\max_k \gamma_{ik}$

---

## 5. 关键点说明

- **样本独立性**：公式中 $x_i$ 明确表示第 $i$ 个样本，联合概率通过连乘 $\prod_{i=1}^N$ 体现
- **隐变量作用**：$z_{ik}$ 显式建模样本归属，通过 $\gamma_{ik}$ 软分配
- **与K-means的关系**：当 $\Sigma_k = \epsilon I$ 且 $\epsilon \to 0$，GMM 退化为 K-means（硬分配 $\gamma_{ik} \in \{0,1\}$）

### 下面我们分析一下GMM和K-means的关系

### 高斯混合模型(GMM)中协方差矩阵极限情况分析

当 $\Sigma_k = \epsilon I$ 且 $\epsilon \to 0$，GMM 退化为 K-means（硬分配 $\gamma_{ik} \in \{0,1\}$）

#### 1. 符号含义
- **$\Sigma_k$**：第$k$个高斯分布的**协方差矩阵**
- **$\epsilon I$**：对角矩阵，对角线元素为$\epsilon$，其余为0（$I$是单位矩阵）
- **$\epsilon \to 0$**：表示$\epsilon$趋近于0的极限过程

#### 2. 数学意义
##### 协方差矩阵的特殊形式
- **$\Sigma_k = \epsilon I$**：
  - 所有高斯成分的协方差矩阵是**各向同性**的（所有方向方差相同）
  - 变量间无相关性（非对角线元素为0）

- **$\epsilon \to 0$**：
  - 高斯分布收缩为**无限窄的尖峰**（方差趋近于0）
  - 概率密度函数趋近于**狄拉克δ函数**（Dirac delta function）
  - 所有概率质量集中在均值$\mu_k$点

#### 3. 在GMM中的具体作用
##### 退化为K-means聚类
当$\epsilon \to 0$时：
1. 每个高斯成分变成**点质量**（仅在均值$\mu_k$处概率非零）
2. 样本$x_i$的隐变量$z_{ik}$会严格分配到**最近的均值$\mu_k$**（硬分配，即K-means行为）
3. 混合系数$\pi_k$退化为簇大小的比例

##### 数学验证
计算责任值$\gamma_{ik}$：
$$
\gamma_{ik} = \frac{\pi_k \mathcal{N}(x_i|\mu_k,\epsilon I)}{\sum_j \pi_j \mathcal{N}(x_i|\mu_j,\epsilon I)}
$$

当$\epsilon \to 0$时：
- 只有距离$x_i$最近的$\mu_k$对应的$\mathcal{N}(x_i|\mu_k,\epsilon I)$非零
- 因此$\gamma_{ik} \in \{0,1\}$（硬分配）

#### 4. 应用场景
1. **理论分析**：说明GMM在协方差矩阵趋近于0时与K-means的等价性
2. **算法设计**：通过调整协方差矩阵的约束形式（如对角、各向同性）控制模型灵活性

#### 5. 总结
该表达式描述了GMM协方差矩阵的极端情况：
- 所有高斯成分的方差趋近于0，分布退化为点质量
- 此时GMM的行为等价于**K-means聚类**（样本被严格分配到最近的簇中心）
